## Evaluation Service

In [ ]:
# Biliotecas necessárias
%pip install -qU langchain-openai
%pip install -qU pypdf langchain_community

### Upload and Read a D&T PDF

In [1]:
import io
import os
import pikepdf
from PyPDF2 import PdfReader
from langchain_community.document_loaders import PyPDFLoader
from langchain.docstore.document import Document
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [29]:
# Função para reparar PDF
def pdf_repair(pdf_content: bytes) -> bytes:
    try:
        # Tentar abrir o PDF e reparar em memória
        with pikepdf.open(io.BytesIO(pdf_content)) as pdf:
            reparado_stream = io.BytesIO()
            pdf.save(reparado_stream)
            return reparado_stream.getvalue()
    except pikepdf.PdfError as e:
        print(f"Erro ao reparar PDF: {e}")
        return None

In [30]:
# Função alternativa para carregar o PDF a partir do BytesIO após reparo
def load_repair_pdf(file_like_object: io.BytesIO):
    try:
        reader = PdfReader(file_like_object)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        
        # Criar manualmente um objeto Document com o texto extraído
        repaired_doc = Document(page_content=text, metadata={"source": "PDF reparado"})
        return [repaired_doc]
    except Exception as e:
        print(f"Erro ao ler o PDF reparado: {e}")
        return None


In [33]:
# Função para carregar o PDF com reparação
def load_pdf(file_path: str):
    try:
        # Tentar carregar o PDF normalmente
        loader = PyPDFLoader(file_path)
        docs = loader.load()
        return docs
    except Exception as e:
        print(f"Erro ao carregar o PDF: {e}")
        
        # Se houver erro, tentar reparar o PDF
        try:
            with open(file_path, "rb") as f:
                pdf_content = f.read()
                pdf_reparado = pdf_repair(pdf_content)
                
                if pdf_reparado:
                    # Carregar o PDF reparado diretamente da memória
                    reparado_stream = io.BytesIO(pdf_reparado)
                    docs_reparados = load_repair_pdf(reparado_stream)
                    
                    if docs_reparados:
                        return docs_reparados
                    else:
                        print("Falha ao ler o PDF reparado.")
                        return None
                else:
                    print("Falha ao reparar o PDF.")
                    return None
        except Exception as e:
            print(f"Erro ao tentar reparar e carregar o PDF: {e}")
            return None


In [35]:
FILE_PATH = "../../data/pdfs/EC/treino/PEGC0558-T.pdf"

docs = load_pdf(FILE_PATH)

if docs:
    print(f"\nPDF carregado com sucesso. Total de documentos: {len(docs)}")


invalid pdf header: b'-----'
incorrect startxref pointer(1)
parsing for Object Streams
Cannot find "/Root" key in trailer
Searching object with "/Catalog" key


Erro ao carregar o PDF: Cannot find Root object in pdf

PDF carregado com sucesso. Total de documentos: 1


In [42]:
print(docs[0].page_content[0:150])
print(docs[0].metadata)

UNIVERSIDADE FEDERAL DE SANTA CATARINA  
PROGRAMA DE PÓS -GRADUAÇÃO EM ENGENHARIA E 
GESTÃO DO CONHECIMENTO  
 
 
 
  EGON SEWALD JUNIOR  
 
 
 
 
 
 
{'source': 'PDF reparado'}


### Model Configuration

In [59]:
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass()


In [60]:
# Função para criar o prompt de análise crítica
def create_analysis_prompt():
    template = """
    Considerando o documento de tese de doutorado que será mostrado a seguir, elabore um resumo considerando as seguintes dimensões de análise:
    a) Originalidade do trabalho;
    b) Relevância para o desenvolvimento científico, tecnológico, cultural e social;
    c) Metodologia utilizada;
    d) Qualidade da redação;
    e) Estrutura/organização do texto;
    f) Interdisciplinaridade.
    Ainda, ao final de cada dimensão analisada, atribua uma nota entre 0 (zero) e 10 (dez).
    
    Documento:
    {content}
    """
    prompt = ChatPromptTemplate.from_template(template)
    return prompt

In [66]:
# Função para criar a Chain com o modelo da OpenAI e o StrOutputParser
def create_analysis_chain():
    # Inicializar o modelo da OpenAI (gpt-4)
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    # Criar o template do prompt
    prompt_template = create_analysis_prompt()
    
    # Criar o parser de saída
    output_parser = StrOutputParser()
    
    # Criar a chain combinando o prompt, o modelo e o parser de saída
    chain = prompt_template | llm | output_parser
    
    return chain

In [67]:
# Carregar o PDF (considerando que 'docs' já tenha sido carregado com o conteúdo do PDF)
if docs:
    pdf_content = docs[0].page_content  # Considerando que o conteúdo do PDF já foi carregado

    # Criar a chain de análise
    analysis_chain = create_analysis_chain()
    
    # Executar a chain passando o conteúdo do PDF
    analysis = analysis_chain.invoke({"content": pdf_content})

    if analysis:
        print("\nAnálise:")
        print(analysis)
    else:
        print("Falha ao gerar a análise.")


Análise:
**Resumo da Tese de Doutorado: "Sistemática para Representação de Conhecimento Judicial Baseado em Colaboração, Consenso e Reputação"**

**a) Originalidade do trabalho:**  
A tese apresenta uma abordagem inovadora ao propor uma sistemática colaborativa para a representação do conhecimento jurídico, utilizando ontologias e integrando conceitos de colaboração, consenso e reputação. A originalidade reside na intersecção entre a Engenharia do Conhecimento e o Direito, abordando a complexidade da linguagem jurídica e a necessidade de um modelo que permita a construção de conhecimento de forma colaborativa, sem a formação de equipes específicas para tal. Nota: 9.

**b) Relevância para o desenvolvimento científico, tecnológico, cultural e social:**  
O trabalho é altamente relevante, pois busca melhorar a eficiência do sistema judiciário, promovendo a celeridade e a assertividade nas decisões judiciais. A proposta de um sistema de conhecimento que apoie magistrados e operadores do d